# Assignment 2 — SSL on Colab
**Before you start:** Runtime → Change runtime type → **T4 GPU**.

1. Upload `ssl-assignment2.zip` to your Google Drive (e.g. `MyDrive/ssl-assignment2.zip`).
3. Run the cells top to bottom. Every result is written to Drive, so if Colab disconnects just re-run Setup and continue from the next experiment.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config (already filled in)

In [ ]:
# Paths are already filled in for your Drive -- no need to change anything
REPO_ZIP  = "/content/drive/MyDrive/ITCS8169/ssl-assignment2.zip"
TRAIN_DIR = "/content/drive/MyDrive/ITCS8169/assignment1/data/train"
TEST_DIR  = "/content/drive/MyDrive/ITCS8169/assignment1/data/test"
DATA_ZIP  = ""
RUNS = "/content/drive/MyDrive/ITCS8169/ssl_runs"   # results are saved here
IMG = 96

## Setup (re-run after every disconnect)

In [ ]:
import os, subprocess, pathlib
os.chdir("/content")
!rm -rf /content/ssl-assignment2 && unzip -q -o "$REPO_ZIP" -d /content
if DATA_ZIP:
    !mkdir -p /content/data_raw && unzip -q -o "$DATA_ZIP" -d /content/data_raw
# copy data to local disk (reading thousands of small files from Drive is slow)
!rm -rf /content/ssl-assignment2/data && mkdir -p /content/ssl-assignment2/data
!cp -r "$TRAIN_DIR" /content/ssl-assignment2/data/train
!cp -r "$TEST_DIR"  /content/ssl-assignment2/data/test
os.makedirs(RUNS, exist_ok=True)
os.chdir("/content/ssl-assignment2")
for split in ["train", "test"]:
    root = pathlib.Path("data") / split
    counts = {d.name: sum(1 for f in d.rglob("*") if f.is_file()) for d in sorted(root.iterdir()) if d.is_dir()}
    print(f"{split}: {len(counts)} classes, {sum(counts.values())} images ->", counts)

Check the output above: you should see **16 classes** and **2,400 train images**. If you see 1 class, your path points one level too high/low.

## Tier 1 — required comparisons
Run one cell at a time. Each run skips itself if its results already exist on Drive.

In [ ]:
# (1) Supervised baseline
if os.path.exists(f"{RUNS}/sup_baseline/results.json"):
    print("already done:", "sup_baseline")
else:
    !python train.py --data_dir data --img_size $IMG --epochs 200 --aug standard --log_test --out "$RUNS/sup_baseline"

In [ ]:
# (2a) SimCLR pretraining (the longest step)
if os.path.exists(f"{RUNS}/simclr_500/results.json"):
    print("already done:", "simclr_500")
else:
    !python pretrain.py --data_dir data --img_size $IMG --method simclr --epochs 500 --out "$RUNS/simclr_500"

In [ ]:
# (2b) Fine-tune the SimCLR encoder
if os.path.exists(f"{RUNS}/simclr_500_ft/results.json"):
    print("already done:", "simclr_500_ft")
else:
    !python train.py --data_dir data --init_from "$RUNS/simclr_500/encoder.pt" --epochs 100 --lr 0.05 --backbone_lr_mult 0.2 --aug standard --log_test --out "$RUNS/simclr_500_ft"

In [ ]:
# (3) Joint CE + SimCLR
if os.path.exists(f"{RUNS}/joint_w0.5/results.json"):
    print("already done:", "joint_w0.5")
else:
    !python train.py --data_dir data --img_size $IMG --epochs 200 --aug standard --ssl_weight 0.5 --log_test --out "$RUNS/joint_w0.5"

In [ ]:
!python summarize.py --runs "$RUNS"

## Tier 3 — best-recipe candidate and its fair control

In [ ]:
if os.path.exists(f"{RUNS}/simclr_1000/results.json"):
    print("already done:", "simclr_1000")
else:
    !python pretrain.py --data_dir data --img_size $IMG --method simclr --epochs 1000 --out "$RUNS/simclr_1000"

In [ ]:
if os.path.exists(f"{RUNS}/best_ft/results.json"):
    print("already done:", "best_ft")
else:
    !python train.py --data_dir data --init_from "$RUNS/simclr_1000/encoder.pt" --epochs 150 --lr 0.05 --backbone_lr_mult 0.3 --aug strong --mix both --label_smoothing 0.1 --log_test --out "$RUNS/best_ft"

In [ ]:
# Same strong recipe without SSL: shows whether the gain is from SSL or from augmentation
if os.path.exists(f"{RUNS}/sup_strong/results.json"):
    print("already done:", "sup_strong")
else:
    !python train.py --data_dir data --img_size $IMG --epochs 300 --aug strong --mix both --label_smoothing 0.1 --log_test --out "$RUNS/sup_strong"

## Tier 2 (optional) — low-label study

In [ ]:
if os.path.exists(f"{RUNS}/sup_lf0.1/results.json"):
    print("already done:", "sup_lf0.1")
else:
    !python train.py --data_dir data --img_size $IMG --epochs 200 --label_frac 0.1 --out "$RUNS/sup_lf0.1"

In [ ]:
if os.path.exists(f"{RUNS}/simclr_ft_lf0.1/results.json"):
    print("already done:", "simclr_ft_lf0.1")
else:
    !python train.py --data_dir data --init_from "$RUNS/simclr_500/encoder.pt" --epochs 100 --lr 0.05 --backbone_lr_mult 0.2 --label_frac 0.1 --out "$RUNS/simclr_ft_lf0.1"

In [ ]:
if os.path.exists(f"{RUNS}/sup_lf0.25/results.json"):
    print("already done:", "sup_lf0.25")
else:
    !python train.py --data_dir data --img_size $IMG --epochs 200 --label_frac 0.25 --out "$RUNS/sup_lf0.25"

In [ ]:
if os.path.exists(f"{RUNS}/simclr_ft_lf0.25/results.json"):
    print("already done:", "simclr_ft_lf0.25")
else:
    !python train.py --data_dir data --init_from "$RUNS/simclr_500/encoder.pt" --epochs 100 --lr 0.05 --backbone_lr_mult 0.2 --label_frac 0.25 --out "$RUNS/simclr_ft_lf0.25"

## Summary table, curves, and checking the best checkpoint

In [ ]:
!python summarize.py --runs "$RUNS"
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{RUNS}/summary.md").read()))
if os.path.exists(f"{RUNS}/curves.png"): display(Image(f"{RUNS}/curves.png"))

In [ ]:
BEST = "best_ft"   # change to whichever run has the highest test accuracy
!python evaluate.py --ckpt "$RUNS/$BEST/model.pt" --data_dir data